# Learned Representation Analysis

Analyze exported seed-0 final-model embeddings. This notebook does **not** encode images locally: it loads `train_embeddings.pt` and `eval_embeddings.pt` produced on the cluster, fits PCA on train embeddings only, projects eval embeddings into that basis.

In [ ]:
from pathlib import Path
import json
import os
import sys


def _find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "experiments").is_dir():
            return candidate
    return start.resolve()


REPO_ROOT = _find_repo_root(Path.cwd())
os.environ.setdefault("MPLCONFIGDIR", str(REPO_ROOT / ".mplconfig"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn.decomposition import PCA

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from plot_style import FULL_WIDTH, apply_matplotlib_style, figure_size, palette
    apply_matplotlib_style()
except Exception:
    FULL_WIDTH = 5.5
    def figure_size(width=FULL_WIDTH, height=None, ratio=0.618):
        return (width, width * ratio if height is None else height)
    palette = {
        "Dark Blue": "tab:blue",
        "Dark Red": "tab:red",
        "Dark Grey": "0.25",
        "Med Grey": "0.65",
    }

EXPORT_ROOT = REPO_ROOT / "experiments" / "embedding_analysis" / "outputs"

# Kept for the figure cells below (paths are unused for writing).
FIG_DIR = REPO_ROOT / "notebooks" / "figures" / "embedding_analysis"
TABLE_DIR = REPO_ROOT / "notebooks" / "results" / "embedding_analysis"

N_VIS = 1600
PCA_COMPONENTS = 32
PCA_SPECTRUM_COMPONENTS = 15
RNG = np.random.default_rng(0)

ENVS = [
    {"name": "TwoRoom", "slug": "tworoom"},
    {"name": "Reacher", "slug": "reacher"},
    {"name": "Push-T", "slug": "pusht"},
    {"name": "OGBench-Cube", "slug": "ogbcube"},
]

METHODS = [
    {"name": "inverse", "label": "ours / inverse"},
    {"name": "sigreg", "label": "sigreg"},
    # {"name": "forward_only", "label": "forward-only"},
]

ACTION_DIMS = {
    "tworoom": 2,
    "reacher": 2,
    "pusht": 2,
    "ogbcube": 5,
}

print(f"Repo root:   {REPO_ROOT}")
print(f"Export root: {EXPORT_ROOT}  (exists: {EXPORT_ROOT.exists()})")

In [ ]:
def torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def as_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def find_split_file(export_dir, split):
    export_dir = Path(export_dir)
    candidates = [
        export_dir / f"{split}_embeddings.pt",
        export_dir / f"{split}.pt",
    ]
    candidates.extend(sorted(export_dir.glob(f"*{split}*embedding*.pt")))
    candidates.extend(sorted(export_dir.glob(f"*{split}*.pt")))
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"No {split} embedding file found in {export_dir}")


def load_split(export_dir, split):
    path = find_split_file(export_dir, split)
    payload = torch_load(path)
    if isinstance(payload, dict) and "embeddings" in payload:
        z = as_numpy(payload["embeddings"]).astype(np.float32)
        arrays = {k: as_numpy(v) for k, v in payload.get("arrays", {}).items()}
        metadata = payload.get("metadata", {})
    else:
        z = as_numpy(payload).astype(np.float32)
        arrays = {}
        metadata = {}
    return {"path": path, "z": z, "arrays": arrays, "metadata": metadata}


def load_export(env_slug, method_name):
    export_dir = EXPORT_ROOT / env_slug / method_name
    train = load_split(export_dir, "train")
    eval_ = load_split(export_dir, "eval")
    keys = sorted(set(train["arrays"]) | set(eval_["arrays"]))
    print(f"\n[{env_slug}/{method_name}]")
    print(f"  train: {train['z'].shape}  {train['path'].relative_to(REPO_ROOT)}")
    print(f"  eval:  {eval_['z'].shape}  {eval_['path'].relative_to(REPO_ROOT)}")
    print(f"  array keys: {keys}")
    return {"train": train, "eval": eval_, "keys": keys, "dir": export_dir}


def choose_rows(n, n_vis=N_VIS):
    if n <= n_vis:
        return np.arange(n)
    return np.sort(RNG.choice(n, size=n_vis, replace=False))


exports = {}
for env in ENVS:
    for method in METHODS:
        try:
            exports[(env["slug"], method["name"])] = load_export(env["slug"], method["name"])
        except FileNotFoundError as exc:
            print(f"[skip] {env['slug']}/{method['name']}: {exc}")

In [ ]:
def component(arrays, key, dim=None):
    if key not in arrays:
        return None
    arr = np.asarray(arrays[key])
    if dim is None:
        return arr.reshape(arr.shape[0], -1)
    arr2 = arr.reshape(arr.shape[0], -1)
    if dim >= arr2.shape[1]:
        return None
    return arr2[:, dim]


def first_available_component(arrays, candidates):
    for label, key, dim in candidates:
        values = component(arrays, key, dim)
        if values is not None:
            return label, values
    return None, None


PLOT_STATE_SPECS = {
    "tworoom": {
        "x": [("agent x", "pos_agent", 0), ("agent x", "proprio", 0), ("agent x", "observation", 0)],
        "y": [("agent y", "pos_agent", 1), ("agent y", "proprio", 1), ("agent y", "observation", 1)],
    },
    "reacher": {
        "x": [("joint angle 1", "qpos", 0), ("joint angle 1", "observation", 0)],
        "y": [("joint angle 2", "qpos", 1), ("joint angle 2", "observation", 1)],
    },
    "pusht": {
        "x": [("agent x", "state", 0), ("agent x", "proprio", 0)],
        "y": [("agent y", "state", 1), ("agent y", "proprio", 1)],
    },
    "ogbcube": {
        "x": [("gripper x", "proprio_effector_pos", 0), ("gripper x", "observation", 0)],
        "y": [("gripper y", "proprio_effector_pos", 1), ("gripper y", "observation", 1)],
    },
}


def get_plot_state(env_slug, arrays):
    spec = PLOT_STATE_SPECS[env_slug]
    x_label, x = first_available_component(arrays, spec["x"])
    y_label, y = first_available_component(arrays, spec["y"])
    if x is None or y is None:
        return None
    return np.stack([x, y], axis=1), (x_label, y_label)


def colors_from_state_xy(state_xy):
    """Encode 2D physical state as hue from x and brightness from y."""
    state_xy = np.asarray(state_xy)
    x = state_xy[:, 0]
    y = state_xy[:, 1]
    xn = (x - np.nanmin(x)) / np.maximum(np.nanmax(x) - np.nanmin(x), 1e-12)
    yn = (y - np.nanmin(y)) / np.maximum(np.nanmax(y) - np.nanmin(y), 1e-12)
    return np.clip(plt.cm.turbo(xn)[:, :3] * (0.45 + 0.55 * yn)[:, None], 0, 1)


def set_square_state_limits(ax, xy):
    xy = np.asarray(xy)
    xmin, ymin = np.nanmin(xy, axis=0)
    xmax, ymax = np.nanmax(xy, axis=0)
    span = max(xmax - xmin, ymax - ymin, 1e-6)
    cx, cy = 0.5 * (xmin + xmax), 0.5 * (ymin + ymax)
    pad = 0.04 * span
    ax.set_xlim(cx - 0.5 * span - pad, cx + 0.5 * span + pad)
    ax.set_ylim(cy - 0.5 * span - pad, cy + 0.5 * span + pad)


def set_symmetric_pc_limits(ax, pc_xy):
    pc_xy = np.asarray(pc_xy)
    radius = np.nanmax(np.abs(pc_xy))
    radius = max(radius, 1e-6)
    radius = np.ceil(2 * radius) / 2
    ax.set_xlim(-radius - 0.2, radius + 0.2)
    ax.set_ylim(-radius - 0.2, radius + 0.2)
    ticks = np.linspace(-radius, radius, 5)
    ax.set_xticks([])
    ax.set_yticks([])


## Protocol Notes

- PCA is fit on train embeddings only; eval embeddings are projected into the train PCA basis.
- The physical-state and encoded-embedding panels are computed on eval embeddings only.


### SensorimotorWM Embeddings

In [ ]:
# Uses only our inverse method and reuses the Stage 1 PCA/state/embedding plotting logic.

import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpecFromSubplotSpec

FINAL_METHOD = {"name": "inverse", "label": "ours / inverse"}
FINAL_SUMMARY_ENVS = [
    {"name": "TwoRoom", "slug": "tworoom", "col_label": "TwoRoom"},
    {"name": "Push-T", "slug": "pusht", "col_label": "Push-T"},
    {"name": "Reacher", "slug": "reacher", "col_label": "Reacher"},
    {"name": "OGBench-Cube", "slug": "ogbcube", "col_label": "Cube"},
]
FINAL_3D_PC_COMBOS = [
    (0, 1, 2),
    (0, 1, 3),
    (0, 2, 3),
    (1, 2, 3),
]

# 2D PC indices to plot for each env (0-based). Cube uses PC1 vs PC3.
PC2D_IDX = {
    "tworoom": (0, 1),
    "pusht": (0, 1),
    "ogbcube": (0, 2),
}

ROW_LABELS = ["", "Physical state", "Encoded embeddings"]


def bilinear_state_colors(state_xy):
    """Map empirical 2D physical-state coordinates to red/green/blue/yellow corner colors."""
    state_xy = np.asarray(state_xy, dtype=np.float32)
    x = state_xy[:, 0]
    y = state_xy[:, 1]

    xn = (x - np.nanmin(x)) / np.maximum(np.nanmax(x) - np.nanmin(x), 1e-12)
    yn = (y - np.nanmin(y)) / np.maximum(np.nanmax(y) - np.nanmin(y), 1e-12)
    xn = np.clip(xn, 0.0, 1.0)
    yn = np.clip(yn, 0.0, 1.0)

    # Physical-state corners:
    # bottom-left  = blue
    # bottom-right = green
    # top-left     = red
    # top-right    = yellow
    c_bl = np.array([0.10, 0.25, 0.90])  # blue
    c_br = np.array([0.05, 0.62, 0.25])  # green
    c_tl = np.array([0.85, 0.10, 0.10])  # red
    c_tr = np.array([0.95, 0.78, 0.10])  # yellow

    colors = (
        (1.0 - xn)[:, None] * (1.0 - yn)[:, None] * c_bl
        + xn[:, None] * (1.0 - yn)[:, None] * c_br
        + (1.0 - xn)[:, None] * yn[:, None] * c_tl
        + xn[:, None] * yn[:, None] * c_tr
    )
    return np.clip(colors, 0.0, 1.0)


if "set_equal_3d_limits" not in globals():
    def set_equal_3d_limits(ax, xyz):
        xyz = np.asarray(xyz)
        mins = np.nanmin(xyz, axis=0)
        maxs = np.nanmax(xyz, axis=0)
        centers = 0.5 * (mins + maxs)
        radius = 0.5 * max(np.nanmax(maxs - mins), 1e-6)
        ax.set_xlim(centers[0] - radius, centers[0] + radius)
        ax.set_ylim(centers[1] - radius, centers[1] + radius)
        ax.set_zlim(centers[2] - radius, centers[2] + radius)


def style_final_3d_axis(ax, xyz, combo):
    set_equal_3d_limits(ax, xyz)
    ax.set_proj_type("ortho")
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=22, azim=-55)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.grid(False)
    ax.set_anchor("C")
    ax.tick_params(axis="both", which="major", pad=-6, length=0)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0))
        axis.pane.set_edgecolor("0.82")
        axis.line.set_color("0.35")
        axis.line.set_linewidth(0.45)
    # Sub-title sits just above the 3D box rather than overlapping its top.
    ax.text2D(
        0.5,
        1.04,
        f"PC{combo[0] + 1}/{combo[1] + 1}/{combo[2] + 1}",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=5.8,
    )


def _format_radians(x, pos=None):
    """Format a tick value as a multiple of pi (e.g. -pi, -pi/2, 0, pi/2, pi)."""
    half_pi = np.pi / 2.0
    n = int(np.round(x / half_pi))
    if abs(x - n * half_pi) > 0.05 * half_pi:
        return f"{x:.2f}"
    if n == 0:
        return "0"
    sign = "-" if n < 0 else ""
    n_abs = abs(n)
    if n_abs == 1:
        return rf"${sign}\pi/2$"
    if n_abs == 2:
        return rf"${sign}\pi$"
    if n_abs % 2 == 0:
        return rf"${sign}{n_abs // 2}\pi$"
    return rf"${sign}{n_abs}\pi/2$"


def apply_radian_ticks(ax, step=np.pi / 2.0, axis="both"):
    locator = mticker.MultipleLocator(step)
    formatter = mticker.FuncFormatter(_format_radians)
    if axis in ("x", "both"):
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(formatter)
    if axis in ("y", "both"):
        ax.yaxis.set_major_locator(locator)
        ax.yaxis.set_major_formatter(formatter)


def set_centered_pc_limits(ax, pc_vis, pad_frac=0.10):
    """Limits centered on the empirical data midpoint (square aspect)."""
    pc_vis = np.asarray(pc_vis)
    x_min, x_max = float(np.nanmin(pc_vis[:, 0])), float(np.nanmax(pc_vis[:, 0]))
    y_min, y_max = float(np.nanmin(pc_vis[:, 1])), float(np.nanmax(pc_vis[:, 1]))
    cx = 0.5 * (x_min + x_max)
    cy = 0.5 * (y_min + y_max)
    half = 0.5 * max(x_max - x_min, y_max - y_min, 1e-6) * (1.0 + pad_frac)
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)


fig = plt.figure(figsize=figure_size(FULL_WIDTH, height=5.4))
gs = fig.add_gridspec(
    3,
    len(FINAL_SUMMARY_ENVS),
    left=0.085,
    right=0.995,
    bottom=0.085,
    top=0.945,
    wspace=0.36,
    hspace=0.28,
)

# Column titles (environments) -- placed just above the top row.
top_bbox = gs[0, 0].get_position(fig)
title_y = top_bbox.y1 + 0.012
for col, env in enumerate(FINAL_SUMMARY_ENVS):
    bbox = gs[0, col].get_position(fig)
    fig.text(
        0.5 * (bbox.x0 + bbox.x1),
        title_y,
        env["col_label"],
        ha="center",
        va="bottom",
        fontsize=plt.rcParams["axes.titlesize"],
    )

# Row labels (panel type) -- placed at the left edge.
for row, label in enumerate(ROW_LABELS):
    row_bbox = gs[row, 0].get_position(fig)
    fig.text(
        0.012,
        0.5 * (row_bbox.y0 + row_bbox.y1),
        label,
        ha="left",
        va="center",
        rotation=90,
        fontsize=plt.rcParams["axes.labelsize"],
    )

for col, env in enumerate(FINAL_SUMMARY_ENVS):
    env_slug = env["slug"]
    key = (env_slug, FINAL_METHOD["name"])

    ax_spec = fig.add_subplot(gs[0, col])
    ax_state = fig.add_subplot(gs[1, col])
    ax_spec.set_box_aspect(1)
    ax_state.set_box_aspect(1)
    # Drop ticks on the physical-state panel.
    ax_state.set_xticks([])
    ax_state.set_yticks([])

    if key not in exports:
        for ax in (ax_spec, ax_state):
            ax.text(0.5, 0.5, "export unavailable", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
        ax_missing = fig.add_subplot(gs[2, col])
        ax_missing.text(0.5, 0.5, "export unavailable", ha="center", va="center", transform=ax_missing.transAxes)
        ax_missing.set_axis_off()
        continue

    export = exports[key]
    z_train = export["train"]["z"]
    z_eval = export["eval"]["z"]
    n_components = min(PCA_COMPONENTS, z_train.shape[1], z_train.shape[0])
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(z_train)
    eval_pc = pca.transform(z_eval)

    # Row 0: PCA spectrum.
    n_spectrum = min(PCA_SPECTRUM_COMPONENTS, len(pca.explained_variance_ratio_))
    xs = np.arange(1, n_spectrum + 1)
    evr = pca.explained_variance_ratio_[:n_spectrum]
    ax_spec.bar(xs, evr, color=palette.get("Dark Blue", "tab:blue"), width=0.52, zorder=2)
    action_dim = ACTION_DIMS.get(env_slug)
    if action_dim is not None and action_dim <= n_spectrum:
        ax_spec.axvline(
            action_dim + 0.5,
            color=palette.get("Dark Red", "tab:red"),
            linestyle="--",
            linewidth=1.0,
            zorder=3,
            label=f"action dim = {action_dim}",
        )
        ax_spec.legend(loc="upper right", handlelength=1.2, fontsize=6)
    ax_spec.set_xlim(0.3, n_spectrum + 0.7)
    ax_spec.set_ylim(0, max(0.05, 1.08 * float(np.max(evr))))
    ax_spec.set_xticks([x for x in [1, 5, 10, 15] if x <= n_spectrum])
    ax_spec.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
    ax_spec.set_xlabel("Principal component")
    if col == 0:
        ax_spec.set_ylabel("Exp. var. ratio")
    else:
        ax_spec.set_ylabel("")
    ax_spec.grid(True, axis="y")

    # Row 1: physical state.
    state = get_plot_state(env_slug, export["eval"]["arrays"])
    if state is None:
        ax_state.text(0.5, 0.5, "state keys unavailable", ha="center", va="center", transform=ax_state.transAxes)
        ax_state.set_axis_off()
        ax_missing = fig.add_subplot(gs[2, col])
        ax_missing.text(0.5, 0.5, "state keys unavailable", ha="center", va="center", transform=ax_missing.transAxes)
        ax_missing.set_axis_off()
        continue

    state_xy, labels = state
    rows = choose_rows(min(len(state_xy), len(eval_pc)))
    state_vis = state_xy[rows]
    pc_vis = eval_pc[rows]

    # Drop a small fraction of state-space outliers that would otherwise
    # skew the bilinear color mapping (compressing most points into a tiny
    # color range). Push-T has a few stray states well outside the table.
    if env_slug == "pusht":
        lo = np.nanpercentile(state_vis, 0.5, axis=0)
        hi = np.nanpercentile(state_vis, 99.5, axis=0)
        keep = np.all((state_vis >= lo) & (state_vis <= hi), axis=1)
        state_vis = state_vis[keep]
        pc_vis = pc_vis[keep]

    colors = bilinear_state_colors(state_vis)

    ax_state.scatter(
        state_vis[:, 0],
        state_vis[:, 1],
        c=colors,
        s=2.0,
        linewidths=0,
        rasterized=True,
    )
    ax_state.set_xlabel(labels[0])
    ax_state.set_ylabel(labels[1])
    ax_state.set_aspect("equal", adjustable="box")
    set_square_state_limits(ax_state, state_vis)
    # Tick-free state panels (re-apply after limits in case helpers reset them).
    ax_state.set_xticks([])
    ax_state.set_yticks([])

    # Row 2: encoded embeddings.
    if env_slug == "reacher":
        if pc_vis.shape[1] < 4:
            ax_pc = fig.add_subplot(gs[2, col])
            ax_pc.text(0.5, 0.5, "need at least 4 PCs", ha="center", va="center", transform=ax_pc.transAxes)
            ax_pc.set_axis_off()
        else:
            inner = GridSpecFromSubplotSpec(
                2, 2,
                subplot_spec=gs[2, col],
                wspace=0.18,
                hspace=-0.05,
            )
            for i, combo in enumerate(FINAL_3D_PC_COMBOS):
                ax3d = fig.add_subplot(inner[i // 2, i % 2], projection="3d")
                ax3d.scatter(
                    pc_vis[:, combo[0]],
                    pc_vis[:, combo[1]],
                    pc_vis[:, combo[2]],
                    c=colors,
                    s=0.3,
                    linewidths=0,
                    depthshade=False,
                    rasterized=True,
                )
                style_final_3d_axis(ax3d, pc_vis[:, combo], combo)
                pos = ax3d.get_position()
                scale = 1.08
                dx = 0.5 * (scale - 1.0) * pos.width
                dy = 0.5 * (scale - 1.0) * pos.height
                ax3d.set_position([pos.x0 - dx, pos.y0 - dy, scale * pos.width, scale * pos.height])
    else:
        idx_x, idx_y = PC2D_IDX[env_slug]
        n_have = pc_vis.shape[1]
        if max(idx_x, idx_y) >= n_have:
            ax_pc = fig.add_subplot(gs[2, col])
            ax_pc.text(
                0.5, 0.5, f"need at least {max(idx_x, idx_y) + 1} PCs",
                ha="center", va="center", transform=ax_pc.transAxes,
            )
            ax_pc.set_axis_off()
        else:
            ax_pc = fig.add_subplot(gs[2, col])
            ax_pc.set_box_aspect(1)
            pc_xy = pc_vis[:, [idx_x, idx_y]]
            ax_pc.scatter(
                pc_xy[:, 0],
                pc_xy[:, 1],
                c=colors,
                s=2.0,
                linewidths=0,
                rasterized=True,
                # alpha=0.9
            )
            ax_pc.set_xlabel(f"PC{idx_x + 1}")
            ax_pc.set_ylabel(f"PC{idx_y + 1}")
            ax_pc.set_aspect("equal", adjustable="box")
            set_centered_pc_limits(ax_pc, pc_xy)
            # Drop ticks on PC scatter plots.
            ax_pc.set_xticks([])
            ax_pc.set_yticks([])

final_summary_path = FIG_DIR / "final_representation_summary.svg"
plt.show()
if "agg" in plt.get_backend().lower():
    plt.close(fig)
else:
    plt.show()

### LeWM (SIGReg) Embeddings

In [ ]:
# Uses only the sigreg method and reuses the Stage 1 PCA/state/embedding plotting logic.

import matplotlib.ticker as mticker
FINAL_METHOD = {"name": "sigreg", "label": "sigreg"}
FINAL_SUMMARY_ENVS = [
    {"name": "TwoRoom", "slug": "tworoom", "col_label": "TwoRoom"},
    {"name": "Push-T", "slug": "pusht", "col_label": "Push-T"},
    {"name": "Reacher", "slug": "reacher", "col_label": "Reacher"},
    {"name": "OGBench-Cube", "slug": "ogbcube", "col_label": "Cube"},
]
ROW_LABELS = ["", "Physical state", "Encoded embeddings"]


def bilinear_state_colors(state_xy):
    """Map empirical 2D physical-state coordinates to red/green/blue/yellow corner colors."""
    state_xy = np.asarray(state_xy, dtype=np.float32)
    x = state_xy[:, 0]
    y = state_xy[:, 1]

    xn = (x - np.nanmin(x)) / np.maximum(np.nanmax(x) - np.nanmin(x), 1e-12)
    yn = (y - np.nanmin(y)) / np.maximum(np.nanmax(y) - np.nanmin(y), 1e-12)
    xn = np.clip(xn, 0.0, 1.0)
    yn = np.clip(yn, 0.0, 1.0)

    # Physical-state corners:
    # bottom-left  = blue
    # bottom-right = green
    # top-left     = red
    # top-right    = yellow
    c_bl = np.array([0.10, 0.25, 0.90])  # blue
    c_br = np.array([0.05, 0.62, 0.25])  # green
    c_tl = np.array([0.85, 0.10, 0.10])  # red
    c_tr = np.array([0.95, 0.78, 0.10])  # yellow

    colors = (
        (1.0 - xn)[:, None] * (1.0 - yn)[:, None] * c_bl
        + xn[:, None] * (1.0 - yn)[:, None] * c_br
        + (1.0 - xn)[:, None] * yn[:, None] * c_tl
        + xn[:, None] * yn[:, None] * c_tr
    )
    return np.clip(colors, 0.0, 1.0)



def set_centered_pc_limits(ax, pc_vis, pad_frac=0.10):
    """Limits centered on the empirical data midpoint (square aspect)."""
    pc_vis = np.asarray(pc_vis)
    x_min, x_max = float(np.nanmin(pc_vis[:, 0])), float(np.nanmax(pc_vis[:, 0]))
    y_min, y_max = float(np.nanmin(pc_vis[:, 1])), float(np.nanmax(pc_vis[:, 1]))
    cx = 0.5 * (x_min + x_max)
    cy = 0.5 * (y_min + y_max)
    half = 0.5 * max(x_max - x_min, y_max - y_min, 1e-6) * (1.0 + pad_frac)
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)


fig = plt.figure(figsize=figure_size(FULL_WIDTH, height=5.4))
gs = fig.add_gridspec(
    3,
    len(FINAL_SUMMARY_ENVS),
    left=0.085,
    right=0.995,
    bottom=0.085,
    top=0.945,
    wspace=0.36,
    hspace=0.28,
)

# Column titles (environments) -- placed just above the top row.
top_bbox = gs[0, 0].get_position(fig)
title_y = top_bbox.y1 + 0.012
for col, env in enumerate(FINAL_SUMMARY_ENVS):
    bbox = gs[0, col].get_position(fig)
    fig.text(
        0.5 * (bbox.x0 + bbox.x1),
        title_y,
        env["col_label"],
        ha="center",
        va="bottom",
        fontsize=plt.rcParams["axes.titlesize"],
    )

# Row labels (panel type) -- placed at the left edge.
for row, label in enumerate(ROW_LABELS):
    row_bbox = gs[row, 0].get_position(fig)
    fig.text(
        0.012,
        0.5 * (row_bbox.y0 + row_bbox.y1),
        label,
        ha="left",
        va="center",
        rotation=90,
        fontsize=plt.rcParams["axes.labelsize"],
    )

for col, env in enumerate(FINAL_SUMMARY_ENVS):
    env_slug = env["slug"]
    key = (env_slug, FINAL_METHOD["name"])

    ax_spec = fig.add_subplot(gs[0, col])
    ax_state = fig.add_subplot(gs[1, col])
    ax_spec.set_box_aspect(1)
    ax_state.set_box_aspect(1)
    # Drop ticks on the physical-state panel.
    ax_state.set_xticks([])
    ax_state.set_yticks([])

    if key not in exports:
        for ax in (ax_spec, ax_state):
            ax.text(0.5, 0.5, "export unavailable", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
        ax_missing = fig.add_subplot(gs[2, col])
        ax_missing.text(0.5, 0.5, "export unavailable", ha="center", va="center", transform=ax_missing.transAxes)
        ax_missing.set_axis_off()
        continue

    export = exports[key]
    z_train = export["train"]["z"]
    z_eval = export["eval"]["z"]
    n_components = min(PCA_COMPONENTS, z_train.shape[1], z_train.shape[0])
    pca = PCA(n_components=n_components, random_state=0)
    pca.fit(z_train)
    eval_pc = pca.transform(z_eval)

    # Row 0: PCA spectrum.
    n_spectrum = min(PCA_SPECTRUM_COMPONENTS, len(pca.explained_variance_ratio_))
    xs = np.arange(1, n_spectrum + 1)
    evr = pca.explained_variance_ratio_[:n_spectrum]
    ax_spec.bar(xs, evr, color=palette.get("Dark Blue", "tab:blue"), width=0.52, zorder=2)
    action_dim = ACTION_DIMS.get(env_slug)
    if action_dim is not None and action_dim <= n_spectrum:
        ax_spec.axvline(
            action_dim + 0.5,
            color=palette.get("Dark Red", "tab:red"),
            linestyle="--",
            linewidth=1.0,
            zorder=3,
            label=f"action dim = {action_dim}",
        )
        ax_spec.legend(loc="upper right", handlelength=1.2, fontsize=6)
    ax_spec.set_xlim(0.3, n_spectrum + 0.7)
    ax_spec.set_ylim(0, max(0.05, 1.08 * float(np.max(evr))))
    ax_spec.set_xticks([x for x in [1, 5, 10, 15] if x <= n_spectrum])
    ax_spec.yaxis.set_major_locator(mticker.MaxNLocator(nbins=3))
    ax_spec.set_xlabel("Principal component")
    if col == 0:
        ax_spec.set_ylabel("Exp. var. ratio")
    else:
        ax_spec.set_ylabel("")
    ax_spec.grid(True, axis="y")

    # Row 1: physical state.
    state = get_plot_state(env_slug, export["eval"]["arrays"])
    if state is None:
        ax_state.text(0.5, 0.5, "state keys unavailable", ha="center", va="center", transform=ax_state.transAxes)
        ax_state.set_axis_off()
        ax_missing = fig.add_subplot(gs[2, col])
        ax_missing.text(0.5, 0.5, "state keys unavailable", ha="center", va="center", transform=ax_missing.transAxes)
        ax_missing.set_axis_off()
        continue

    state_xy, labels = state
    rows = choose_rows(min(len(state_xy), len(eval_pc)))
    state_vis = state_xy[rows]
    pc_vis = eval_pc[rows]

    # Drop a small fraction of state-space outliers that would otherwise
    # skew the bilinear color mapping (compressing most points into a tiny
    # color range). Push-T has a few stray states well outside the table.
    if env_slug == "pusht":
        lo = np.nanpercentile(state_vis, 0.5, axis=0)
        hi = np.nanpercentile(state_vis, 99.5, axis=0)
        keep = np.all((state_vis >= lo) & (state_vis <= hi), axis=1)
        state_vis = state_vis[keep]
        pc_vis = pc_vis[keep]

    colors = bilinear_state_colors(state_vis)

    ax_state.scatter(
        state_vis[:, 0],
        state_vis[:, 1],
        c=colors,
        s=2.0,
        linewidths=0,
        rasterized=True,
    )
    ax_state.set_xlabel(labels[0])
    ax_state.set_ylabel(labels[1])
    ax_state.set_aspect("equal", adjustable="box")
    set_square_state_limits(ax_state, state_vis)
    # Tick-free state panels (re-apply after limits in case helpers reset them).
    ax_state.set_xticks([])
    ax_state.set_yticks([])

    # Row 2: encoded embeddings (PC1 vs PC2 for every environment).
    idx_x, idx_y = 0, 1
    n_have = pc_vis.shape[1]
    if max(idx_x, idx_y) >= n_have:
        ax_pc = fig.add_subplot(gs[2, col])
        ax_pc.text(
            0.5, 0.5, f"need at least {max(idx_x, idx_y) + 1} PCs",
            ha="center", va="center", transform=ax_pc.transAxes,
        )
        ax_pc.set_axis_off()
    else:
        ax_pc = fig.add_subplot(gs[2, col])
        ax_pc.set_box_aspect(1)
        pc_xy = pc_vis[:, [idx_x, idx_y]]
        ax_pc.scatter(
            pc_xy[:, 0],
            pc_xy[:, 1],
            c=colors,
            s=2.0,
            linewidths=0,
            rasterized=True,
        )
        ax_pc.set_xlabel(f"PC{idx_x + 1}")
        ax_pc.set_ylabel(f"PC{idx_y + 1}")
        ax_pc.set_aspect("equal", adjustable="box")
        set_centered_pc_limits(ax_pc, pc_xy)
        # Drop ticks on PC scatter plots.
        ax_pc.set_xticks([])
        ax_pc.set_yticks([])

for suffix in ("svg", "pdf"):
    final_summary_path = FIG_DIR / f"final_representation_summary_sigreg.{suffix}"
    plt.show()
if "agg" in plt.get_backend().lower():
    plt.close(fig)
else:
    plt.show()